# Tanzania GDP Prediction using Machine Learning
## Linear Regression vs Decision Tree Regressor

**Objective**: Predict Tanzania's GDP using economic indicators

**Models**:
1. Linear Regression
2. Decision Tree Regressor

## 1. Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ All libraries imported successfully!")

## 2. Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('tanzania_gdp_data.csv')

print("Dataset Shape:", df.shape)
print("\n" + "="*50)
print("First 5 rows:")
df.head()

In [ ]:
# Dataset information
print("Dataset Information:")
print("="*50)
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary:")
print("="*50)
df.describe()

In [ ]:
# Check for missing values
print("Missing Values Analysis:")
print("="*50)
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Percentage': missing_percent
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(missing_df)

## 3. Data Visualization

In [ ]:
# GDP Trend over time
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['GDP_Billion_USD'], linewidth=2, color='#2E86AB')
plt.title('Tanzania GDP Trend (2000-2024)', fontsize=16, fontweight='bold')
plt.xlabel('Time Period (Quarterly)', fontsize=12)
plt.ylabel('GDP (Billion USD)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of GDP
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df['GDP_Billion_USD'], bins=30, color='#A23B72', edgecolor='black', alpha=0.7)
plt.title('GDP Distribution', fontsize=14, fontweight='bold')
plt.xlabel('GDP (Billion USD)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)

plt.subplot(1, 2, 2)
plt.boxplot(df['GDP_Billion_USD'], vert=True, patch_artist=True,
            boxprops=dict(facecolor='#F18F01', alpha=0.7),
            medianprops=dict(color='red', linewidth=2))
plt.title('GDP Box Plot', fontsize=14, fontweight='bold')
plt.ylabel('GDP (Billion USD)', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (top features)
# Select numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
correlation = df[numeric_cols].corr()

# Get top correlated features with GDP
gdp_corr = correlation['GDP_Billion_USD'].abs().sort_values(ascending=False)[1:11]
top_features = gdp_corr.index.tolist()
top_features.insert(0, 'GDP_Billion_USD')

plt.figure(figsize=(12, 10))
sns.heatmap(df[top_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, linewidths=1, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix - Top 10 Features with GDP', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 10 Features Correlated with GDP:")
print("="*50)
print(gdp_corr)

## 4. Data Preprocessing

In [ ]:
# Handle missing values - Using mean imputation
print("Handling Missing Values...")
print("="*50)

# Create a copy for preprocessing
df_clean = df.copy()

# Fill missing values with column mean
for column in df_clean.columns:
    if df_clean[column].isnull().sum() > 0:
        df_clean[column].fillna(df_clean[column].mean(), inplace=True)

print(f"Missing values after imputation: {df_clean.isnull().sum().sum()}")
print("✓ Missing values handled successfully!")

In [ ]:
# Prepare features (X) and target (y)
print("\nPreparing Features and Target Variable...")
print("="*50)

# Drop non-predictive columns and target variable
X = df_clean.drop(['GDP_Billion_USD', 'Year', 'Quarter'], axis=1)
y = df_clean['GDP_Billion_USD']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

In [ ]:
# Split data into training and testing sets (80-20 split)
print("\nSplitting Data...")
print("="*50)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"\nTraining set: {X_train.shape[0]/len(X)*100:.1f}%")
print(f"Testing set: {X_test.shape[0]/len(X)*100:.1f}%")

In [ ]:
# Feature Scaling (important for Linear Regression)
print("\nFeature Scaling...")
print("="*50)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled successfully!")
print(f"Mean of scaled training features: {X_train_scaled.mean():.4f}")
print(f"Std of scaled training features: {X_train_scaled.std():.4f}")

## 5. Model 1: Linear Regression

In [ ]:
print("=" * 60)
print("MODEL 1: LINEAR REGRESSION")
print("=" * 60)

# Create and train the model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lr_train = lr_model.predict(X_train_scaled)
y_pred_lr_test = lr_model.predict(X_test_scaled)

print("✓ Linear Regression model trained successfully!")

In [ ]:
# Evaluate Linear Regression
print("\nLinear Regression Performance:")
print("="*50)

# Training metrics
lr_train_r2 = r2_score(y_train, y_pred_lr_train)
lr_train_mae = mean_absolute_error(y_train, y_pred_lr_train)
lr_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_lr_train))

# Testing metrics
lr_test_r2 = r2_score(y_test, y_pred_lr_test)
lr_test_mae = mean_absolute_error(y_test, y_pred_lr_test)
lr_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr_test))

print("TRAINING SET:")
print(f"  R² Score: {lr_train_r2:.4f}")
print(f"  MAE: ${lr_train_mae:.4f} Billion")
print(f"  RMSE: ${lr_train_rmse:.4f} Billion")

print("\nTESTING SET:")
print(f"  R² Score: {lr_test_r2:.4f}")
print(f"  MAE: ${lr_test_mae:.4f} Billion")
print(f"  RMSE: ${lr_test_rmse:.4f} Billion")

In [ ]:
# Feature importance for Linear Regression (coefficients)
feature_importance_lr = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nTop 10 Most Important Features (Linear Regression):")
print("="*50)
print(feature_importance_lr.head(10))

# Visualize top features
plt.figure(figsize=(12, 6))
top_10 = feature_importance_lr.head(10)
colors = ['green' if x > 0 else 'red' for x in top_10['Coefficient']]
plt.barh(top_10['Feature'], top_10['Coefficient'], color=colors, alpha=0.7)
plt.xlabel('Coefficient Value', fontsize=12)
plt.title('Top 10 Feature Importance - Linear Regression', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted - Linear Regression
plt.figure(figsize=(14, 6))

# Training set
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_pred_lr_train, alpha=0.5, color='blue')
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel('Actual GDP (Billion USD)', fontsize=11)
plt.ylabel('Predicted GDP (Billion USD)', fontsize=11)
plt.title(f'Training Set - Linear Regression\nR² = {lr_train_r2:.4f}', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)

# Testing set
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_lr_test, alpha=0.5, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual GDP (Billion USD)', fontsize=11)
plt.ylabel('Predicted GDP (Billion USD)', fontsize=11)
plt.title(f'Testing Set - Linear Regression\nR² = {lr_test_r2:.4f}', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Model 2: Decision Tree Regressor

In [ ]:
print("=" * 60)
print("MODEL 2: DECISION TREE REGRESSOR")
print("=" * 60)

# Create and train the model (no need for scaling)
dt_model = DecisionTreeRegressor(max_depth=10, min_samples_split=10, random_state=42)
dt_model.fit(X_train, y_train)

# Make predictions
y_pred_dt_train = dt_model.predict(X_train)
y_pred_dt_test = dt_model.predict(X_test)

print("✓ Decision Tree model trained successfully!")
print(f"Tree depth: {dt_model.get_depth()}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")

In [ ]:
# Evaluate Decision Tree
print("\nDecision Tree Performance:")
print("="*50)

# Training metrics
dt_train_r2 = r2_score(y_train, y_pred_dt_train)
dt_train_mae = mean_absolute_error(y_train, y_pred_dt_train)
dt_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_dt_train))

# Testing metrics
dt_test_r2 = r2_score(y_test, y_pred_dt_test)
dt_test_mae = mean_absolute_error(y_test, y_pred_dt_test)
dt_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_dt_test))

print("TRAINING SET:")
print(f"  R² Score: {dt_train_r2:.4f}")
print(f"  MAE: ${dt_train_mae:.4f} Billion")
print(f"  RMSE: ${dt_train_rmse:.4f} Billion")

print("\nTESTING SET:")
print(f"  R² Score: {dt_test_r2:.4f}")
print(f"  MAE: ${dt_test_mae:.4f} Billion")
print(f"  RMSE: ${dt_test_rmse:.4f} Billion")

In [ ]:
# Feature importance for Decision Tree
feature_importance_dt = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features (Decision Tree):")
print("="*50)
print(feature_importance_dt.head(10))

# Visualize top features
plt.figure(figsize=(12, 6))
top_10_dt = feature_importance_dt.head(10)
plt.barh(top_10_dt['Feature'], top_10_dt['Importance'], color='#E63946', alpha=0.7)
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 10 Feature Importance - Decision Tree', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted - Decision Tree
plt.figure(figsize=(14, 6))

# Training set
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_pred_dt_train, alpha=0.5, color='purple')
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel('Actual GDP (Billion USD)', fontsize=11)
plt.ylabel('Predicted GDP (Billion USD)', fontsize=11)
plt.title(f'Training Set - Decision Tree\nR² = {dt_train_r2:.4f}', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)

# Testing set
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_dt_test, alpha=0.5, color='orange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual GDP (Billion USD)', fontsize=11)
plt.ylabel('Predicted GDP (Billion USD)', fontsize=11)
plt.title(f'Testing Set - Decision Tree\nR² = {dt_test_r2:.4f}', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Model Comparison

In [ ]:
# Create comparison dataframe
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree'],
    'Train_R2': [lr_train_r2, dt_train_r2],
    'Test_R2': [lr_test_r2, dt_test_r2],
    'Train_MAE': [lr_train_mae, dt_train_mae],
    'Test_MAE': [lr_test_mae, dt_test_mae],
    'Train_RMSE': [lr_train_rmse, dt_train_rmse],
    'Test_RMSE': [lr_test_rmse, dt_test_rmse]
})

print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)
print(comparison.to_string(index=False))
print("="*70)

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['R2', 'MAE', 'RMSE']
colors_lr = ['#2E86AB', '#A23B72']
colors_dt = ['#F18F01', '#C73E1D']

# R² Score
axes[0].bar(['LR Train', 'LR Test'], [lr_train_r2, lr_test_r2], color=colors_lr, alpha=0.7, label='Linear Regression')
axes[0].bar(['DT Train', 'DT Test'], [dt_train_r2, dt_test_r2], color=colors_dt, alpha=0.7, label='Decision Tree')
axes[0].set_ylabel('R² Score', fontsize=11)
axes[0].set_title('R² Score Comparison', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')

# MAE
axes[1].bar(['LR Train', 'LR Test'], [lr_train_mae, lr_test_mae], color=colors_lr, alpha=0.7, label='Linear Regression')
axes[1].bar(['DT Train', 'DT Test'], [dt_train_mae, dt_test_mae], color=colors_dt, alpha=0.7, label='Decision Tree')
axes[1].set_ylabel('MAE (Billion USD)', fontsize=11)
axes[1].set_title('Mean Absolute Error Comparison', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# RMSE
axes[2].bar(['LR Train', 'LR Test'], [lr_train_rmse, lr_test_rmse], color=colors_lr, alpha=0.7, label='Linear Regression')
axes[2].bar(['DT Train', 'DT Test'], [dt_train_rmse, dt_test_rmse], color=colors_dt, alpha=0.7, label='Decision Tree')
axes[2].set_ylabel('RMSE (Billion USD)', fontsize=11)
axes[2].set_title('Root Mean Squared Error Comparison', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Residual Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Linear Regression residuals
lr_residuals_train = y_train - y_pred_lr_train
lr_residuals_test = y_test - y_pred_lr_test

# Decision Tree residuals
dt_residuals_train = y_train - y_pred_dt_train
dt_residuals_test = y_test - y_pred_dt_test

# LR Training residuals
axes[0, 0].scatter(y_pred_lr_train, lr_residuals_train, alpha=0.5, color='blue')
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted GDP', fontsize=10)
axes[0, 0].set_ylabel('Residuals', fontsize=10)
axes[0, 0].set_title('Linear Regression - Training Residuals', fontsize=11, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# LR Testing residuals
axes[0, 1].scatter(y_pred_lr_test, lr_residuals_test, alpha=0.5, color='green')
axes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Predicted GDP', fontsize=10)
axes[0, 1].set_ylabel('Residuals', fontsize=10)
axes[0, 1].set_title('Linear Regression - Testing Residuals', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# DT Training residuals
axes[1, 0].scatter(y_pred_dt_train, dt_residuals_train, alpha=0.5, color='purple')
axes[1, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Predicted GDP', fontsize=10)
axes[1, 0].set_ylabel('Residuals', fontsize=10)
axes[1, 0].set_title('Decision Tree - Training Residuals', fontsize=11, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# DT Testing residuals
axes[1, 1].scatter(y_pred_dt_test, dt_residuals_test, alpha=0.5, color='orange')
axes[1, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Predicted GDP', fontsize=10)
axes[1, 1].set_ylabel('Residuals', fontsize=10)
axes[1, 1].set_title('Decision Tree - Testing Residuals', fontsize=11, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Final Conclusion

In [ ]:
print("\n" + "="*70)
print("FINAL CONCLUSION")
print("="*70)

# Determine best model
if lr_test_r2 > dt_test_r2:
    best_model = "Linear Regression"
    best_r2 = lr_test_r2
    best_mae = lr_test_mae
else:
    best_model = "Decision Tree"
    best_r2 = dt_test_r2
    best_mae = dt_test_mae

print(f"\n🏆 BEST PERFORMING MODEL: {best_model}")
print(f"   Test R² Score: {best_r2:.4f}")
print(f"   Test MAE: ${best_mae:.4f} Billion")

print("\n📊 KEY INSIGHTS:")
print("   • Linear Regression is good for understanding linear relationships")
print("   • Decision Tree can capture non-linear patterns in the data")
print("   • Both models show strong predictive capability for Tanzania's GDP")

print("\n💡 RECOMMENDATIONS:")
print("   • Consider ensemble methods (Random Forest, XGBoost) for better accuracy")
print("   • Perform hyperparameter tuning for Decision Tree")
print("   • Include more recent economic indicators if available")
print("   • Monitor model performance over time and retrain as needed")

print("\n" + "="*70)
print("✓ Analysis Complete!")
print("="*70)

## 9. Save Predictions (Optional)

In [ ]:
# Create predictions dataframe
predictions_df = pd.DataFrame({
    'Actual_GDP': y_test.values,
    'LR_Predicted': y_pred_lr_test,
    'DT_Predicted': y_pred_dt_test,
    'LR_Error': y_test.values - y_pred_lr_test,
    'DT_Error': y_test.values - y_pred_dt_test
})

print("\nSample Predictions:")
print(predictions_df.head(10))

# Optionally save to CSV
# predictions_df.to_csv('gdp_predictions.csv', index=False)
# print("\n✓ Predictions saved to 'gdp_predictions.csv'")